In [12]:
import numpy as np
import feos
import si_units as si

# --- setup ---
parameters = feos.Parameters.from_json(
    ["hydrogen", "nitrogen", "argon"],
    "parameters.json"
)
pcsaft = feos.EquationOfState.pcsaft(parameters)

# --- example ---
z1 = 0.7
z2 = 0.1
z3 = 1 - z1 - z2
z = np.array([z1, z2, z3])
T = 100                           # K
P = 20e5                          # Pa (50 bar)

eq = feos.PhaseEquilibrium.tp_flash(pcsaft, T * si.KELVIN, P* si.PASCAL, z*si.MOL)

eq.liquid.molefracs


array([0.11399248, 0.17846119, 0.70754633])

In [7]:
import numpy as np
import feos
import si_units as si

# --- setup ---
parameters = feos.Parameters.from_json(
    ["argon", "nitrogen", "hydrogen"],
    "parameters.json"
)
pcsaft = feos.EquationOfState.pcsaft(parameters)

def tp_flash_xy(pcsaft, T_K, P_Pa, z):
    """
    TP flash for a ternary mixture.
    Returns (x, y, beta_v) where beta_v is vapor fraction if available, else None.
    Raises an exception if flash fails or single-phase (depending on FeOS behavior).
    """
    z = np.asarray(z, dtype=float)
    z = z / z.sum()  # normalize just in case

    eq = feos.PhaseEquilibrium.tp_flash(
        pcsaft,
        T_K * si.KELVIN,
        P_Pa * si.PASCAL,
        z * si.MOL)

    # two-phase compositions
    x = eq.liquid.molefracs()
    y = eq.vapor.molefracs()

    # Some FeOS builds expose phase fraction; if not, ignore.
    beta_v = None
    for name in ("vapor_fraction", "beta_v", "vapor_phase_fraction"):
        if hasattr(eq, name):
            beta_v = getattr(eq, name)()
            break

    return x, y, beta_v


# --- example single call ---
z = np.array([0.8, 0.00005, 0.19995])   # Ar, N2, H2
T = 100                           # K
P = 20e5                          # Pa (20 bar)

try:
    x, y, beta_v = tp_flash_xy(pcsaft, T, P, z)
    print("x (liquid) =", x)
    print("y (vapor)  =", y)
    if beta_v is not None:
        print("vapor fraction =", beta_v)
except Exception as e:
    print("Flash failed or single-phase at this (T,P,z):", e)


Flash failed or single-phase at this (T,P,z): 'numpy.ndarray' object is not callable


In [2]:
z = np.array([0.95, 0.04, 0.01]) # CH4, N2, CO2

temperatures = np.arange(220.0, 281.0, 10.0)   # K
pressures_bar = np.array([10, 20, 40, 60])     # bar

results = []

for T in temperatures:
    for Pbar in pressures_bar:
        P = Pbar * 1e5  # Pa

        try:
            x, y, beta_v = tp_flash_xy(pcsaft, T, P, z)
            results.append({
                "T_K": T,
                "P_bar": Pbar,
                "z": z.copy(),
                "x": np.array(x, dtype=float),
                "y": np.array(y, dtype=float),
                "beta_v": None if beta_v is None else float(beta_v),
            })
            print(f"OK  T={T:6.1f} K  P={Pbar:6.1f} bar  x={x}  y={y}")

        except Exception as e:
            print(f"NO  T={T:6.1f} K  P={Pbar:6.1f} bar  reason: {e}")
            continue


NO  T= 220.0 K  P=  10.0 bar  reason: No phase split according to stability analysis.
NO  T= 220.0 K  P=  20.0 bar  reason: No phase split according to stability analysis.
NO  T= 220.0 K  P=  40.0 bar  reason: No phase split according to stability analysis.
NO  T= 220.0 K  P=  60.0 bar  reason: No phase split according to stability analysis.
NO  T= 230.0 K  P=  10.0 bar  reason: No phase split according to stability analysis.
NO  T= 230.0 K  P=  20.0 bar  reason: No phase split according to stability analysis.
NO  T= 230.0 K  P=  40.0 bar  reason: No phase split according to stability analysis.
NO  T= 230.0 K  P=  60.0 bar  reason: No phase split according to stability analysis.
NO  T= 240.0 K  P=  10.0 bar  reason: No phase split according to stability analysis.
NO  T= 240.0 K  P=  20.0 bar  reason: No phase split according to stability analysis.
NO  T= 240.0 K  P=  40.0 bar  reason: No phase split according to stability analysis.
NO  T= 240.0 K  P=  60.0 bar  reason: No phase split a